# Limit LSS covariance matrix check
The GPY test requires to compute the covariance matrix of the LSSs in the limit where N,M tends to infinity at some specified rate M/N=c.

The computation of this matrix is not trivial, so this notebook is used as an additional check. 

# Empirical covariance

In [6]:
# check that the sample test statistics follow the same covariance
from gpy_test import GPY
from gpy_test.result import GPYResult

from statsmodels.tsa.arima_process import arma_generate_sample
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm


# the complex gaussian time series will be ARMA(1,1) with the following parameters
ar = 0
ma = 0
N = 500  # number of samples
M = 1000  # number of time series
burn= 100

c = N / M

# for the GPY test, we will use the following test functions
fs = [lambda x: x, lambda x: x**3]

In [7]:
def run() -> GPYResult:
    real = arma_generate_sample([1, -ar], [1, ma], (N+burn, M), scale=1)
    y = real 
    # Provide a dummy covariance matrix, it won't be used. Here we are just interested in
    # collecting samples of the test statistics to estimate their covariance
    Cov = np.identity(len(fs))  
    return GPY(y, fs, Cov)


# Generate a large number of experiments
n_repeat = 1000
results = Parallel(n_jobs=-1)(delayed(run)() for _ in tqdm(range(n_repeat)))

# show the empirical covariance of the test statistics
np.cov(np.array([result.lss for result in results]).T)

100%|██████████| 1000/1000 [00:40<00:00, 24.92it/s]


array([[   4.67146008,   84.76680303],
       [  84.76680303, 2025.07293251]])

In [8]:
def run() -> GPYResult:
    real = arma_generate_sample([1, -ar], [1, ma], (N+burn, M), scale=1 / np.sqrt(2))
    imag = arma_generate_sample([1, -ar], [1, ma], (N+burn, M), scale=1 / np.sqrt(2))
    y = real + 1j * imag
    # Provide a dummy covariance matrix, it won't be used. Here we are just interested in
    # collecting samples of the test statistics to estimate their covariance
    Cov = np.identity(len(fs))  
    return GPY(y[burn:], fs, Cov)


# Generate a large number of experiments
n_repeat = 1000
results = Parallel(n_jobs=-1)(delayed(run)() for _ in tqdm(range(n_repeat)))

# show the empirical covariance of the test statistics
np.cov(np.array([result.lss for result in results]).T)

100%|██████████| 1000/1000 [01:43<00:00,  9.69it/s]


array([[  2.02012254,  30.81620396],
       [ 30.81620396, 614.92360611]])

# Limit Covariance

In [9]:
from gpy_test.config.covariance import CovarianceConfig
from gpy_test.covariance import covariance

# we also need to know the range of the eigenvalues of the covariance matrix. For that we just use
# the range of the eigenvalues of the sample covariance matrix for one of the experiments
eig_range = np.min(results[0].eigs_S_1)-1, np.max(results[0].eigs_S_1)+1
# eig_range = (0, 4)

# we will also need to provide the spectral density of the ARMA(1,1) process. It can also be computed
# from the sample time series.
def _ARMA_spectral_density(ar: float, ma: float) -> callable:
    ma_part = lambda nu: 1 + ma**2 + 2 * ma * np.cos(2 * np.pi * nu)
    ar_part = lambda nu: 1 / (1 + ar**2 - 2 * ar * np.cos(2 * np.pi * nu))
    return lambda nu: ar_part(nu) * ma_part(nu)

oracle_sd = _ARMA_spectral_density(ar, ma)

# dblquad integration
Using dblquad is more precise (and precision is configurable) but can be quite expensive to run.

In [10]:
# load the base config
d = {'integral_config': {'type_': 'dblquad',
  'n_points': None,
  'epsabs': 1e-1,
  'epsrel': 1e-1},
 'fixed_point_config': {'init_m_real': 1.0,
  'init_m_imag': 1.0,
  'max_steps': 100,
  'tolerance': 1e-3},
 'contour_config_pair': ({'imag_height': 0.2,
   'real_slack': 0.2,
   'type_': 'circle'},
  {'imag_height': 0.1, 'real_slack': 0.1, 'type_': 'circle'}),
 'derivative_epsilon': 1e-08,
 'admissible_imag': 0.01,
 'n_jobs': -1,
 'verbose': True}
covariance_config = CovarianceConfig(**d)

# compute the covariance
covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 2624.72it/s]

array([[  2.00000535,  16.49996139],
       [ 16.49996139, 177.907566  ]])

In [10]:
# load the base config
d = {'integral_config': {'type_': 'dblquad',
  'n_points': None,
  'epsabs': 1e-1,
  'epsrel': 1e-1},
 'fixed_point_config': {'init_m_real': 1.0,
  'init_m_imag': 1.0,
  'max_steps': 100,
  'tolerance': 1e-3},
 'contour_config_pair': ({'imag_height': 0.2,
   'real_slack': 0.2,
   'type_': 'circle'},
  {'imag_height': 0.1, 'real_slack': 0.1, 'type_': 'circle'}),
 'derivative_epsilon': 1e-08,
 'admissible_imag': 0.01,
 'n_jobs': -1,
 'verbose': True}
covariance_config = CovarianceConfig(**d)

# compute the covariance
covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 1823.61it/s]


array([[ 1.99999962,  5.99999107],
       [ 5.99999107, 20.00000548]])

# Simpson integration
Can be much faster as we evaluate upfront the integrand on a fixed grid, but error is not controllable. 

In [6]:
d = base_covariance_config.dict()

# make sure integration sample is dblquad
d['integral_config']["type_"] = "dblsimpson"
d['integral_config']["n_points"] = 100
d['integral_config']["epsabs"] = None
d['integral_config']["epsrel"] = None

# make the contour be an ellipse
d["contour_config_pair"][0]["type_"] = "ellipse"
d["contour_config_pair"][1]["type_"] = "ellipse"

# set the config and check the values
covariance_config = CovarianceConfig(**d)
print(covariance_config)

# compute the covariance
covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

NameError: name 'base_covariance_config' is not defined

In [ ]:
d = base_covariance_config.dict()

# make sure integration sample is dblquad
d['integral_config']["type_"] = "dblsimpson"
d['integral_config']["n_points"] = 100
d['integral_config']["epsabs"] = None
d['integral_config']["epsrel"] = None

# make the contour be an ellipse
d["contour_config_pair"][0]["type_"] = "circle"
d["contour_config_pair"][1]["type_"] = "circle"

# set the config and check the values
covariance_config = CovarianceConfig(**d)
print(covariance_config)

# compute the covariance
covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

integral_config=IntegralConfig(type_='dblsimpson', n_points=100, epsabs=None, epsrel=None) fixed_point_config=FixedPointConfig(init_m_real=1.0, init_m_imag=1.0, max_steps=100, tolerance=0.01) contour_config_pair=(ContourConfig(imag_height=0.2, real_slack=0.2, type_='circle'), ContourConfig(imag_height=0.1, real_slack=0.1, type_='circle')) derivative_epsilon=1e-08 admissible_imag=0.001 n_jobs=-1 verbose=True


100%|██████████| 3/3 [00:00<00:00, 1673.26it/s]


ValueError: Convergence issue: the limit covariance matrix is not definite non-negative: covariance=array([[ 1.50775571,  4.6606286 ],
       [ 4.6606286 , 12.87255151]])

In [ ]:
d = base_covariance_config.dict()

# make sure integration sample is dblquad
d['integral_config']["type_"] = "dblsimpson"
d['integral_config']["n_points"] = 100
d['integral_config']["epsabs"] = None
d['integral_config']["epsrel"] = None

# make the contour be an ellipse
d["contour_config_pair"][0]["type_"] = "rectangle" # we can try the non C1 rectangle contour here since we are using dblsimpson
d["contour_config_pair"][1]["type_"] = "rectangle"

# set the config and check the values
covariance_config = CovarianceConfig(**d)
print(covariance_config)

# compute the covariance
covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

integral_config=IntegralConfig(type_='dblsimpson', n_points=100, epsabs=None, epsrel=None) fixed_point_config=FixedPointConfig(init_m_real=1.0, init_m_imag=1.0, max_steps=100, tolerance=0.01) contour_config_pair=(ContourConfig(imag_height=0.2, real_slack=0.2, type_='rectangle'), ContourConfig(imag_height=0.1, real_slack=0.1, type_='rectangle')) derivative_epsilon=1e-08 admissible_imag=0.001 n_jobs=-1 verbose=True


100%|██████████| 3/3 [00:00<00:00, 1057.83it/s]


array([[0.80586591, 1.9172857 ],
       [1.9172857 , 5.09556935]])

# Spectral density as fixed grid estimates
Instead of providing the full spectral density as a Callable, it is also possible to pass the function evaluated on a grid. Can be faster as the function does not need to computed again and again, more at the cost of non-controllable precision. 

In [ ]:
d = base_covariance_config.dict()

# make sure integration sample is dblquad
d['integral_config']["type_"] = "dblquad"
d['integral_config']["n_points"] = None
d['integral_config']["epsabs"] = 1e-1
d['integral_config']["epsrel"] = 1e-1

# make the contour be an ellipse
d["contour_config_pair"][0]["type_"] = "circle"
d["contour_config_pair"][1]["type_"] = "circle"

# set the config and check the values
covariance_config = CovarianceConfig(**d)
print(covariance_config)

# pre-compute the spectral density for the ARMA(1,1) process on a fixed grid
nus = np.linspace(-0.5, 0.5, 50)
sd_values = np.array([oracle_sd(nu) for nu in nus])

# compute the covariance
covariance(
    covariance_config,
    fs,
    (nus, sd_values),
    eig_range,
    c,
)

integral_config=IntegralConfig(type_='dblquad', n_points=None, epsabs=0.1, epsrel=0.1) fixed_point_config=FixedPointConfig(init_m_real=1.0, init_m_imag=1.0, max_steps=100, tolerance=0.01) contour_config_pair=(ContourConfig(imag_height=0.2, real_slack=0.2, type_='circle'), ContourConfig(imag_height=0.1, real_slack=0.1, type_='circle')) derivative_epsilon=1e-08 admissible_imag=0.001 n_jobs=-1 verbose=True


100%|██████████| 3/3 [00:00<00:00, 631.45it/s]


array([[0.7999981 , 1.91999534],
       [1.91999534, 4.92801147]])

In [ ]:
d = base_covariance_config.dict()

# make sure integration sample is dblquad
d['integral_config']["type_"] = "dblsimpson"
d['integral_config']["n_points"] = 100
d['integral_config']["epsabs"] = None
d['integral_config']["epsrel"] = None

# make the contour be an ellipse
d["contour_config_pair"][0]["type_"] = "circle"
d["contour_config_pair"][1]["type_"] = "circle"

# set the config and check the values
covariance_config = CovarianceConfig(**d)
print(covariance_config)

# pre-compute the spectral density for the ARMA(1,1) process on a fixed grid
nus = np.linspace(-0.5, 0.5, 50)
sd_values = np.array([oracle_sd(nu) for nu in nus])

# compute the covariance
covariance(
    covariance_config,
    fs,
    (nus, sd_values),
    eig_range,
    c,
)

integral_config=IntegralConfig(type_='dblsimpson', n_points=100, epsabs=None, epsrel=None) fixed_point_config=FixedPointConfig(init_m_real=1.0, init_m_imag=1.0, max_steps=100, tolerance=0.01) contour_config_pair=(ContourConfig(imag_height=0.2, real_slack=0.2, type_='circle'), ContourConfig(imag_height=0.1, real_slack=0.1, type_='circle')) derivative_epsilon=1e-08 admissible_imag=0.001 n_jobs=-1 verbose=True


100%|██████████| 3/3 [00:00<00:00, 1419.07it/s]


ValueError: Convergence issue: the limit covariance matrix is not definite non-negative: covariance=array([[ 1.5077557 ,  4.66062866],
       [ 4.66062866, 12.87255131]])